In [ ]:
#Cell 1 — Mount Google Drive
import os
from google.colab import drive

# Ensure the mount point is clean before attempting to mount
if os.path.exists('/content/drive'):
    # Attempt to unmount first if it was somehow mounted incorrectly
    try:
        drive.flush_and_unmount()
    except ValueError:
        pass # Ignore if not mounted
    # Then remove the directory to ensure it's empty
    os.system('rm -rf /content/drive/*')

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip -q install transformers datasets evaluate scikit-learn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [ ]:
%%writefile train_tinybert_agnews_colab.py
import os
import json
import time
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from tqdm.auto import tqdm


@dataclass
class Config:
    # ===== Model / Data =====
    model_name: str = "huawei-noah/TinyBERT_General_4L_312D"
    dataset_name: str = "ag_news"
    setting: str = "centralized_full_finetuning"
    num_labels: int = 4
    seed: int = 42

    # ===== Paths =====
    drive_root: str = "/content/drive/MyDrive/tinybert_agnews_centralized"
    experiment_name: str = "tinybert_full_finetuning_agnews"

    # ===== Data =====
    max_length: int = 256
    validation_ratio: float = 0.1

    # ===== Training =====
    train_batch_size: int = 32
    eval_batch_size: int = 64
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    max_epochs: int = 100
    warmup_ratio: float = 0.1

    # ===== Early stopping =====
    monitor: str = "accuracy"      # hoặc "macro_f1"
    greater_is_better: bool = True
    early_stopping_patience: int = 3
    min_delta: float = 0.0

    # ===== System =====
    num_workers: int = 2
    fp16: bool = True
    resume: bool = True
    save_latest_k: int = 2

    @property
    def exp_dir(self):
        return os.path.join(self.drive_root, self.experiment_name)

    @property
    def ckpt_dir(self):
        return os.path.join(self.exp_dir, "checkpoints")

    @property
    def best_model_dir(self):
        return os.path.join(self.exp_dir, "best_model")

    @property
    def final_model_dir(self):
        return os.path.join(self.exp_dir, "final_model")

    @property
    def history_json_path(self):
        return os.path.join(self.exp_dir, "history.json")

    @property
    def history_csv_path(self):
        return os.path.join(self.exp_dir, "history.csv")

    @property
    def summary_path(self):
        return os.path.join(self.exp_dir, "run_summary.json")

    @property
    def config_path(self):
        return os.path.join(self.exp_dir, "config.json")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dirs(cfg: Config):
    os.makedirs(cfg.drive_root, exist_ok=True)
    os.makedirs(cfg.exp_dir, exist_ok=True)
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    os.makedirs(cfg.best_model_dir, exist_ok=True)
    os.makedirs(cfg.final_model_dir, exist_ok=True)


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_history_csv(path, history):
    if len(history) == 0:
        return
    pd.DataFrame(history).to_csv(path, index=False, encoding="utf-8")


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params


def tokenize_function(examples, tokenizer, max_length):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
    )


def prepare_data(cfg: Config, tokenizer):
    ds = load_dataset(cfg.dataset_name)

    split_ds = ds["train"].train_test_split(
        test_size=cfg.validation_ratio,
        seed=cfg.seed,
        shuffle=True
    )

    train_ds = split_ds["train"]
    val_ds = split_ds["test"]
    test_ds = ds["test"]

    tokenized_train = train_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.max_length),
        batched=True,
        remove_columns=["text"],
    )
    tokenized_val = val_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.max_length),
        batched=True,
        remove_columns=["text"],
    )
    tokenized_test = test_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.max_length),
        batched=True,
        remove_columns=["text"],
    )

    tokenized_train = tokenized_train.rename_column("label", "labels")
    tokenized_val = tokenized_val.rename_column("label", "labels")
    tokenized_test = tokenized_test.rename_column("label", "labels")

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_loader = DataLoader(
        tokenized_train,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        collate_fn=collator,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        tokenized_val,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    test_loader = DataLoader(
        tokenized_test,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader


def create_model(cfg: Config):
    return AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=cfg.num_labels
    )


def prune_old_checkpoints(ckpt_dir: str, keep_latest_k: int = 2):
    ckpts = [
        os.path.join(ckpt_dir, f)
        for f in os.listdir(ckpt_dir)
        if f.startswith("epoch_") and f.endswith(".pt")
    ]
    ckpts.sort(key=lambda x: int(os.path.basename(x).split("_")[1].split(".")[0]))
    for p in ckpts[:-keep_latest_k]:
        try:
            os.remove(p)
        except Exception:
            pass


def latest_checkpoint_path(ckpt_dir: str):
    if not os.path.exists(ckpt_dir):
        return None
    ckpts = [
        os.path.join(ckpt_dir, f)
        for f in os.listdir(ckpt_dir)
        if f.startswith("epoch_") and f.endswith(".pt")
    ]
    if not ckpts:
        return None
    ckpts.sort(key=lambda x: int(os.path.basename(x).split("_")[1].split(".")[0]))
    return ckpts[-1]


def save_checkpoint(cfg, epoch, model, optimizer, scheduler, scaler, best_metric, patience_counter, history):
    ckpt_path = os.path.join(cfg.ckpt_dir, f"epoch_{epoch:03d}.pt")
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_metric": best_metric,
        "patience_counter": patience_counter,
        "history": history,
        "config": asdict(cfg),
    }
    torch.save(payload, ckpt_path)

    with open(os.path.join(cfg.ckpt_dir, "latest_checkpoint.txt"), "w", encoding="utf-8") as f:
        f.write(ckpt_path)

    prune_old_checkpoints(cfg.ckpt_dir, cfg.save_latest_k)
    return ckpt_path


def load_latest_checkpoint(cfg, model, optimizer=None, scheduler=None, scaler=None, map_location="cpu"):
    ckpt_path = latest_checkpoint_path(cfg.ckpt_dir)
    if ckpt_path is None or not cfg.resume:
        return 0, None, 0, []

    print(f"Resuming from checkpoint: {ckpt_path}")
    payload = torch.load(ckpt_path, map_location=map_location)

    model.load_state_dict(payload["model_state_dict"])
    if optimizer is not None and payload.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(payload["optimizer_state_dict"])
    if scheduler is not None and payload.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(payload["scheduler_state_dict"])
    if scaler is not None and payload.get("scaler_state_dict") is not None:
        scaler.load_state_dict(payload["scaler_state_dict"])

    return (
        payload["epoch"],
        payload.get("best_metric", None),
        payload.get("patience_counter", 0),
        payload.get("history", []),
    )


@torch.no_grad()
def evaluate(model, loader, device, split_name="validation"):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    total_steps = 0

    for batch in tqdm(loader, desc=f"Evaluating {split_name}", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(batch["labels"].cpu().numpy().tolist())
        total_loss += loss.item()
        total_steps += 1

    return {
        "split": split_name,
        "eval_loss": float(total_loss / max(total_steps, 1)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
        "macro_f1": float(f1_score(all_labels, all_preds, average="macro")),
    }


def save_model_and_tokenizer(model, tokenizer, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)


def is_improved(current_metric, best_metric, greater_is_better=True, min_delta=0.0):
    if greater_is_better:
        return current_metric > (best_metric + min_delta)
    return current_metric < (best_metric - min_delta)


def train():
    cfg = Config()
    set_seed(cfg.seed)
    ensure_dirs(cfg)
    save_json(cfg.config_path, asdict(cfg))

    device = get_device()
    print("Device:", device)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    model = create_model(cfg).to(device)

    total_params, trainable_params = count_parameters(model)

    train_loader, val_loader, test_loader = prepare_data(cfg, tokenizer)

    total_training_steps = len(train_loader) * cfg.max_epochs
    warmup_steps = int(total_training_steps * cfg.warmup_ratio)

    optimizer = AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps
    )

    use_amp = cfg.fp16 and device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    start_epoch, best_metric, patience_counter, history = load_latest_checkpoint(
        cfg, model, optimizer, scheduler, scaler, map_location=device
    )

    if best_metric is None:
        best_metric = -1e18 if cfg.greater_is_better else 1e18

    print(f"Start from epoch {start_epoch + 1}")
    print(f"Best {cfg.monitor} so far: {best_metric:.6f}")
    print(f"Patience counter: {patience_counter}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

    stopped_early = False

    for epoch in range(start_epoch + 1, cfg.max_epochs + 1):
        model.train()
        total_loss = 0.0
        total_steps = 0
        epoch_start_time = time.time()

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.max_epochs}")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(**batch)
                loss = outputs.loss

            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            scheduler.step()

            total_loss += loss.item()
            total_steps += 1
            pbar.set_postfix(train_loss=f"{loss.item():.4f}")

        time_per_epoch = time.time() - epoch_start_time
        avg_train_loss = total_loss / max(total_steps, 1)

        val_metrics = evaluate(model, val_loader, device, split_name="validation")
        current_metric = val_metrics[cfg.monitor]

        improved = is_improved(
            current_metric=current_metric,
            best_metric=best_metric,
            greater_is_better=cfg.greater_is_better,
            min_delta=cfg.min_delta
        )

        if improved:
            best_metric = current_metric
            patience_counter = 0
            save_model_and_tokenizer(model, tokenizer, cfg.best_model_dir)
            is_new_best = True
        else:
            patience_counter += 1
            is_new_best = False

        epoch_result = {
            "epoch": int(epoch),
            "train_loss": float(avg_train_loss),
            "eval_loss": float(val_metrics["eval_loss"]),
            "accuracy": float(val_metrics["accuracy"]),
            "macro_f1": float(val_metrics["macro_f1"]),
            "time_per_epoch": float(time_per_epoch),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": cfg.model_name,
            "dataset_name": cfg.dataset_name,
            "setting": cfg.setting,
            "best_metric_so_far": float(best_metric),
            "monitor_metric": cfg.monitor,
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }

        history.append(epoch_result)
        save_json(cfg.history_json_path, history)
        save_history_csv(cfg.history_csv_path, history)

        ckpt_path = save_checkpoint(
            cfg, epoch, model, optimizer, scheduler, scaler,
            best_metric=best_metric,
            patience_counter=patience_counter,
            history=history
        )

        print("-" * 80)
        print(json.dumps(epoch_result, indent=2))
        print("Checkpoint saved:", ckpt_path)

        if patience_counter >= cfg.early_stopping_patience:
            print(f"Early stopping triggered: no improvement in {cfg.early_stopping_patience} consecutive epochs.")
            stopped_early = True
            break

    save_model_and_tokenizer(model, tokenizer, cfg.final_model_dir)

    best_model = AutoModelForSequenceClassification.from_pretrained(
        cfg.best_model_dir,
        num_labels=cfg.num_labels
    ).to(device)
    best_test_metrics = evaluate(best_model, test_loader, device, split_name="test")

    run_summary = {
        "model_name": cfg.model_name,
        "dataset_name": cfg.dataset_name,
        "setting": cfg.setting,
        "monitor_metric": cfg.monitor,
        "best_metric_so_far": float(best_metric),
        "min_delta": cfg.min_delta,
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "max_epochs": int(cfg.max_epochs),
        "num_epochs_completed": int(len(history)),
        "stopped_early": bool(stopped_early),
        "best_model_dir": cfg.best_model_dir,
        "final_model_dir": cfg.final_model_dir,
        "latest_checkpoint": latest_checkpoint_path(cfg.ckpt_dir),
        "test_metrics_using_best_model": best_test_metrics,
    }
    save_json(cfg.summary_path, run_summary)

    print("=" * 80)
    print("Final model saved to:", cfg.final_model_dir)
    print("Best model saved to:", cfg.best_model_dir)
    print("History JSON:", cfg.history_json_path)
    print("History CSV:", cfg.history_csv_path)
    print("Run summary:", cfg.summary_path)
    print("Best-model test metrics:")
    print(json.dumps(best_test_metrics, indent=2))
    print("Done.")


@torch.no_grad()
def load_latest_model_for_inference(exp_dir, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ckpt_dir = os.path.join(exp_dir, "checkpoints")
    ckpt_path = latest_checkpoint_path(ckpt_dir)
    if ckpt_path is None:
        raise FileNotFoundError("No checkpoint found.")

    payload = torch.load(ckpt_path, map_location="cpu")
    cfg_dict = payload["config"]
    model_name = cfg_dict["model_name"]
    num_labels = cfg_dict["num_labels"]

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    model.load_state_dict(payload["model_state_dict"])
    model.to(device)
    model.eval()

    return tokenizer, model, device, ckpt_path


@torch.no_grad()
def predict_text(text, tokenizer, model, device, max_length=256):
    inputs = tokenizer(text, truncation=True, max_length=max_length, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=-1).item()

    id2label = {
        0: "World",
        1: "Sports",
        2: "Business",
        3: "Sci/Tech",
    }
    return pred, id2label[pred]


if __name__ == "__main__":
    train()

Overwriting train_tinybert_agnews_colab.py


In [ ]:
!python train_tinybert_agnews_colab.py

Device: cuda
Loading weights: 100% 71/71 [00:00<00:00, 1903.46it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                   